# Projeto Fictus | Analise de Vendas — Bloco 5: Cenários e Recomendação

---

## Pergunta Central do Bloco
> **Sob quais condições o negócio poderia continuar existindo após a aquisição?**

---

## Contexto do Bloco

Este bloco consolida as descobertas de toda a jornada analítica para responder à pergunta final: sob quais condições o negócio é viável? Através da construção de cenários fundamentados nos dados extraídos (Base, Corte e Otimização), testamos a margem de segurança e o potencial de recuperação de valor do ativo sob uma nova gestão.

O resultado é uma síntese executiva que traduz o comportamento dos dados em uma recomendação estratégica de investimento, ponderando o crescimento observado contra os riscos e gargalos operacionais identificados.

Este bloco não projeta o futuro — projeta consequências. Três cenários são construídos com premissas explícitas, e cada um responde a uma pergunta diferente para o board:

- **Cenário Base:** se nada mudar, o negócio se sustenta?
- **Cenário de Corte:** o que pode ser descontinuado sem comprometer a sobrevivência?
- **Cenário de Otimização:** qual o impacto de resolver o problema logístico — conectando com a Fase 2 — Logística?

O bloco encerra com o **Relatório Final de Recomendação** — consolidando os cinco blocos em uma decisão estruturada para o board.

---

## Nota Metodológica — Deslocamento Temporal
Dados originais Olist **2016–2018** deslocados **+7 anos** → período **2023–2025**.  
**Análise restrita a partir de janeiro/2024** — dados de 2023 desconsiderados.

---

In [ ]:
## Configuração do Ambiente
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
from pathlib import Path

# ─── Caminhos relativos — funcionam em qualquer máquina ─────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_EXT      = BASE_DIR / "data" / "externos"
DIR_EXPORTS  = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)


warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")

## Carregamento dos Dados

In [ ]:
# ─── Separador padrão: vírgula, decimal ponto ────────────────────────────────
#   Todos os arquivos gerados pelo ETL usam sep=',' e decimal='.'
def ler_csv(caminho, sep=",", **kwargs):
    df = pd.read_csv(caminho, sep=sep, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de datas ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega",
            "data_envio_transportadora", "data_aprovacao"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

# ─── Colunas numéricas: garantia adicional ────────────────────────────────────
for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento e filtro ─────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto",  how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente",  how="left")
fato = fato.merge(dim_v[["id_vendedor", "estado_vendedor"]],        on="id_vendedor", how="left")
fato = fato.merge(
    dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]],
    on="id_data", how="left"
)
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

DATA_INICIO = "2024-01-01"
fato = fato[fato["data_compra"] >= DATA_INICIO].copy()

fe  = fato[fato["status_pedido"] == "entregue"].copy()
fne = fato[fato["status_pedido"] != "entregue"].copy()
periodos_ord = sorted(fato["periodo"].dropna().unique())

print(f"[FILTRO] Dados a partir de: {DATA_INICIO}")
print(f"fato (filtrado)   : {len(fato):>8} linhas | {fato['data_compra'].min().date()} → {fato['data_compra'].max().date()}")
print(f"fato_entregues    : {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}% do total)")

# ─── Métricas base para os cenários ──────────────────────────────────────────
receita_total     = fe["preco"].sum()
frete_total       = fe["valor_frete"].sum()
n_pedidos_total   = fe["id_pedido"].nunique()
pct_frete         = frete_total / receita_total * 100
ticket_medio      = receita_total / n_pedidos_total
pct_no_prazo      = fe["entregue_no_prazo"].mean() * 100
nota_media_global = fe["nota_review"].mean()

print(f"Receita total         : R$ {receita_total:,.0f}")
print(f"Frete total           : R$ {frete_total:,.0f} ({pct_frete:.1f}% da receita)")
print(f"Pedidos entregues     : {n_pedidos_total:,}")
print(f"Ticket médio          : R$ {ticket_medio:,.2f}")
print(f"% Entregue no prazo   : {pct_no_prazo:.1f}%")
print(f"Nota média            : {nota_media_global:.2f}")



---

## Análise 1 — Cenário Base: Se nada mudar, o negócio se sustenta?

---

### Premissas do Cenário Base

| Premissa | Valor assumido | Justificativa |
|---|---|---|
| Crescimento de receita | +5% a.a. | Conservador — abaixo da média histórica observada |
| % Frete sobre receita | Constante | Sem intervenção logística |
| Taxa de cancelamento | Constante | Sem melhoria operacional |
| Saída de sellers críticos | Não ocorre | Hipótese de continuidade |
| Investimento pós-aquisição | Nenhum adicional | Operação no estado atual |

> *"A projeção da tendência atual responde a uma pergunta concreta para o board: se nada mudar após a aquisição, o negócio se sustenta? As premissas são conservadoras e declaradas — o objetivo é quantificar o custo de inação, não projetar crescimento otimista."*

**Framework:** PDCA — análise de tendência e custo de inação  
**Entrega:** Projeção de receita e frete nos próximos 4 trimestres com a tendência atual  

**Como este script responde à pergunta:**
> Para responder se o negócio se sustenta sem intervenção, o script projeta as tendências atuais para os próximos 4 trimestres. Calcula o crescimento médio de receita observado no histórico, o % de frete médio e a taxa de cancelamento, e extrapola esses valores mantendo as mesmas proporções — o chamado "custo de inação". Essa é a linha de base contra a qual os outros cenários serão comparados.
>
> 1. **Projeção de receita trimestral:** Plota a receita histórica e estende a linha de tendência para os próximos 4 trimestres com o crescimento médio observado. A área sombreada representa o intervalo de confiança da projeção — mais largo no futuro, refletindo a incerteza crescente. Se a linha projetada mantém trajetória positiva, o negócio sobrevive no curto prazo sem intervenção.
> 2. **Projeção do % de frete:** Estende a tendência do % de frete sobre receita. Se a tendência é crescente, o gráfico mostra em qual trimestre o frete ultrapassa o limiar crítico — o ponto onde a barreira de conversão se torna insustentável. Isso torna o custo de inação visível e datado, não apenas abstrato.

**Análise do Resultado:**
 Este cenário projeta a inércia: como a operação evolui se nada mudar após a aquisição. Se a tendência de receita se mantém positiva e o % de frete não ultrapassa o limiar crítico no horizonte projetado, o negócio sobrevive no curto prazo sem intervenção. Se a tendência de frete é crescente, o gráfico torna visível em qual trimestre a pressão sobre a conversão se torna crítica — o custo de inação datado e quantificado.

In [ ]:
receita_trim = (
    fe.groupby("periodo")
    .agg(
        receita      = ("preco",       "sum"),
        frete_total  = ("valor_frete", "sum"),
        n_pedidos    = ("id_pedido",   "nunique"),
        nota_media   = ("nota_review", "mean"),
        pct_no_prazo = ("entregue_no_prazo", "mean"),
    )
    .reset_index().sort_values("periodo")
)
receita_trim["pct_frete"] = receita_trim["frete_total"] / receita_trim["receita"] * 100
receita_trim["pct_no_prazo"] = pd.to_numeric(receita_trim["pct_no_prazo"], errors="coerce") * 100

# Regressão linear sobre receita trimestral → tendência
x_hist = np.arange(len(receita_trim))
m_rec, b_rec, r_rec, _, _ = np.polyfit(x_hist, receita_trim["receita"].values, 1, full=False), \
                              0, 0, 0, 0
m_rec2, b_rec2 = np.polyfit(x_hist, receita_trim["receita"].values, 1)
m_frete2, b_frete2 = np.polyfit(x_hist, receita_trim["pct_frete"].values, 1)

# Projeção 4 trimestres à frente
N_PROJ  = 4
x_proj  = np.arange(len(receita_trim), len(receita_trim) + N_PROJ)
rec_proj   = m_rec2 * x_proj + b_rec2
frete_proj = np.clip(m_frete2 * x_proj + b_frete2, 10, 60)

# Monta df de projeção
ultimos = [f"P+{i+1}T" for i in range(N_PROJ)]
df_proj = pd.DataFrame({
    "periodo":  ultimos,
    "receita":  rec_proj,
    "pct_frete": frete_proj,
    "projetado": True,
})
receita_trim["projetado"] = False
df_base = pd.concat([receita_trim[["periodo","receita","pct_frete","projetado"]], df_proj],
                    ignore_index=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Cenário 1 — BASE: Projeção com Tendência Atual (sem intervenção)",
             fontsize=13, fontweight="bold")

x_all = range(len(df_base))
cores_rec = [COR_RECEITA if not p else "#7FB3D3" for p in df_base["projetado"]]
axes[0].bar(x_all, df_base["receita"] / 1000, color=cores_rec, alpha=0.85)
axes[0].axvline(len(receita_trim) - 0.5, color="black", linewidth=1.5, linestyle="--", alpha=0.5)
axes[0].text(len(receita_trim), df_base["receita"].max() / 1000 * 0.95,
             "← Histórico | Projetado →", fontsize=8, ha="center", color=COR_NEUTRO)
axes[0].set_title("Receita Trimestral — Histórico + Projeção (R$ mil)", fontsize=11)
axes[0].set_ylabel("Receita (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
axes[0].set_xticks(x_all)
axes[0].set_xticklabels(df_base["periodo"], rotation=45, ha="right", fontsize=7)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_RECEITA, label="Histórico"),
    mpatches.Patch(color="#7FB3D3",   label="Projetado (cenário base)"),
], frameon=False, fontsize=8)

cores_frete = [COR_NEUTRO if not p else COR_ALERTA for p in df_base["projetado"]]
axes[1].plot(range(len(receita_trim)), receita_trim["pct_frete"],
             color=COR_NEUTRO, linewidth=2, marker="o", markersize=4, label="Histórico")
axes[1].plot(range(len(receita_trim) - 1, len(df_base)),
             list(receita_trim["pct_frete"].iloc[-1:]) + list(frete_proj),
             color=COR_ALERTA, linewidth=2, linestyle="--", marker="s", markersize=4, label="Projetado")
axes[1].axhline(pct_frete, color=COR_NEUTRO, linestyle=":", linewidth=1,
                label=f"Média atual: {pct_frete:.1f}%")
axes[1].set_title("% Frete sobre Receita — Tendência Projetada", fontsize=11)
axes[1].set_ylabel("Frete / Receita (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xticks(x_all)
axes[1].set_xticklabels(df_base["periodo"], rotation=45, ha="right", fontsize=7)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "01_cenario_base")
plt.show()

delta_rec_base = (rec_proj[-1] - receita_trim["receita"].iloc[-1]) / receita_trim["receita"].iloc[-1] * 100
print("\n" + "="*60)
print("INSIGHT — CENÁRIO BASE")
print("="*60)
print(f"Receita último trimestre  : R$ {receita_trim['receita'].iloc[-1]:,.0f}")
print(f"Receita projetada (+4T)   : R$ {rec_proj[-1]:,.0f} ({delta_rec_base:+.1f}%)")
print(f"% Frete projetado (+4T)   : {frete_proj[-1]:.1f}%")
veredicto_base = "POSITIVO" if delta_rec_base > 0 and frete_proj[-1] < 25 else \
                 "ALERTA" if delta_rec_base > 0 else "NEGATIVO"
print(f"Veredicto                 : {veredicto_base}")

---

## Análise 2 — Cenário de Corte: O que pode ser descontinuado sem comprometer a sobrevivência?

---

### Premissas do Cenário de Corte

| Premissa | Valor assumido | Justificativa |
|---|---|---|
| Categorias descontinuadas | Q4 (deterioração) + Q2 com frete > 40% | Categorias que destroem valor |
| Impacto na receita | Perda proporcional ao peso das categorias cortadas | Conservador — sem efeito de substituição |
| Ganho de margem | Redução do % frete médio | Menos categorias de alto frete |
| Sellers afetados | Proporcional ao corte de categorias | Estimativa conservadora |

> *"Categorias que destroem mais valor do que geram são candidatas ao corte. A simulação separa o core do portfólio — o que sustenta a receita com eficiência — do periférico — o que consome frete e complexidade operacional sem retorno proporcional."*

**Framework:** Análise de portfólio — identificação do core vs periférico  
**Entrega:** Simulação do impacto do corte de categorias não rentáveis na receita, na margem e na estrutura do portfólio  

**Como este script responde à pergunta:**
> O script identifica categorias candidatas ao corte com base em dois critérios objetivos: estar no quadrante Q4 (receita caindo e frete piorando) ou ter % de frete acima de 40% com baixa representatividade na receita. Para cada categoria candidata, calcula o impacto do corte na receita total, na margem e no % de frete médio do portfólio restante.
>
> 1. **Candidatas ao corte e impacto na receita:** Lista as categorias identificadas com seu peso na receita e % de frete atual. O gráfico mostra quanto da receita seria sacrificado e quanto o portfólio ficaria mais enxuto — a troca entre eficiência e volume.
> 2. **Portfólio antes e depois do corte:** Compara a composição do portfólio nos dois cenários — número de categorias, receita total, % de frete médio e mix de quadrantes. Se o corte remove categorias que representam pouca receita mas muito frete, o portfólio restante tem melhor qualidade mesmo sendo menor.

**Análise do Resultado:**
 Nem todo faturamento é bom. Esta análise simula o que acontece se descontinuarmos as categorias que dão prejuízo, os vendedores problemáticos ou as rotas logísticas mais caras. Para o comprador, o Cenário de Corte responde uma pergunta prática: quanto da receita atual é estruturalmente saudável? Se o corte remove categorias que representam pouca receita mas alto % de frete, o portfólio restante opera com melhor eficiência — e a tese de aquisição fica mais defensável.


In [ ]:
cat_metricas = (
    fe.groupby("nome_categoria_produto")
    .agg(
        receita      = ("preco",       "sum"),
        frete_total  = ("valor_frete", "sum"),
        n_itens      = ("id_pedido",   "count"),
        nota_media   = ("nota_review", "mean"),
    )
    .reset_index()
)
cat_metricas["pct_receita"] = cat_metricas["receita"] / cat_metricas["receita"].sum() * 100
cat_metricas["pct_frete"]   = cat_metricas["frete_total"] / cat_metricas["receita"] * 100
cat_metricas["liquido"]     = cat_metricas["receita"] - cat_metricas["frete_total"]
cat_metricas["pct_liquido"] = cat_metricas["liquido"] / cat_metricas["receita"] * 100

# Categorias candidatas ao corte: frete > 35% OU lucro líquido negativo
cats_corte = cat_metricas[
    (cat_metricas["pct_frete"] > 35) | (cat_metricas["liquido"] < 0)
].copy()
cats_mantidas = cat_metricas[~cat_metricas["nome_categoria_produto"].isin(cats_corte["nome_categoria_produto"])]

receita_pos_corte  = cats_mantidas["receita"].sum()
frete_pos_corte    = cats_mantidas["frete_total"].sum()
pct_frete_pos_corte = frete_pos_corte / receita_pos_corte * 100
pct_receita_perdida = cats_corte["pct_receita"].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Cenário 2 — CORTE: Descontinuação de Categorias Não Rentáveis",
             fontsize=13, fontweight="bold")

# Comparação: antes e depois do corte
comparacao = pd.DataFrame({
    "Métrica":    ["Receita (R$ mil)", "% Frete / Receita", "Nº Categorias"],
    "Antes":      [receita_total/1000, pct_frete,           len(cat_metricas)],
    "Após Corte": [receita_pos_corte/1000, pct_frete_pos_corte, len(cats_mantidas)],
})

x_comp = range(len(comparacao))
width  = 0.35
bars1 = axes[0].bar([xi - width/2 for xi in x_comp], comparacao["Antes"],
                    width, color=COR_NEUTRO, alpha=0.85, label="Antes do corte")
bars2 = axes[0].bar([xi + width/2 for xi in x_comp], comparacao["Após Corte"],
                    width, color=COR_RECEITA, alpha=0.85, label="Após corte")
axes[0].set_xticks(x_comp)
axes[0].set_xticklabels(comparacao["Métrica"], fontsize=9)
axes[0].set_title("Impacto do Corte nas Métricas Principais", fontsize=11)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{bar.get_height():,.1f}", ha="center", va="bottom", fontsize=8)
for bar in bars2:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f"{bar.get_height():,.1f}", ha="center", va="bottom", fontsize=8, color=COR_RECEITA)
axes[0].legend(frameon=False, fontsize=8)

# Categorias candidatas ao corte
cats_corte_sorted = cats_corte.sort_values("pct_frete", ascending=True)
cores_corte = [COR_ALERTA if v < 0 else COR_DESTAQUE for v in cats_corte_sorted["pct_liquido"]]
axes[1].barh(
    [c.replace("_", " ")[:28] for c in cats_corte_sorted["nome_categoria_produto"]],
    cats_corte_sorted["pct_frete"],
    color=cores_corte, alpha=0.85
)
axes[1].axvline(35, color=COR_ALERTA, linestyle="--", linewidth=1, label="Limiar corte: 35%")
for i, (_, row) in enumerate(cats_corte_sorted.iterrows()):
    axes[1].text(row["pct_frete"] + 0.3, i,
                 f"{row['pct_receita']:.1f}% receita",
                 va="center", fontsize=7, color=COR_NEUTRO)
axes[1].set_xlabel("% Frete sobre Receita")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title(f"Categorias Candidatas ao Corte ({len(cats_corte)})", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "02_cenario_corte")
plt.show()

print("\n" + "="*60)
print("INSIGHT — CENÁRIO DE CORTE")
print("="*60)
print(f"Categorias candidatas ao corte : {len(cats_corte)}")
print(f"Receita sacrificada            : {pct_receita_perdida:.1f}%")
print(f"% Frete antes do corte         : {pct_frete:.1f}%")
print(f"% Frete após o corte           : {pct_frete_pos_corte:.1f}%")
print(f"Ganho de eficiência            : {pct_frete - pct_frete_pos_corte:.1f}pp")
veredicto_corte = "VIÁVEL" if pct_receita_perdida < 15 else "ATENÇÃO — perda relevante"
print(f"Veredicto                      : {veredicto_corte}")

---

## Análise 3 — Cenário de Otimização: Qual o impacto de resolver o problema logístico?

---

### Premissas do Cenário de Otimização

| Premissa | Valor assumido | Justificativa |
|---|---|---|
| Redução do frete ao cliente | -20% sobre o valor médio atual | Hipótese conservadora de internalização logística (Fase 2 — Logística) |
| Impacto no volume | +10% de pedidos | Frete menor aumenta conversão — estimativa conservadora |
| Melhoria no SLA | +8pp no % no prazo | Operação própria = maior controle |
| Prazo de implementação | 6 meses | Hipótese operacional |
| Custo de implementação | Não modelado neste bloco | Investigado na Fase 2 — Logística |

> *"O cenário de otimização materializa em números o impacto da Fase 2 — Logística: o que muda na receita, no frete e no SLA se o modelo logístico for otimizado. As premissas são conservadoras e declaradas — o objetivo é mostrar o potencial de valor, não garantir o resultado."*

**Framework:** Planejamento por Cenários + Matriz de Decisão  
**Entrega:** Simulação do impacto da redução de frete no volume, na receita e na margem operacional  

**Como este script responde à pergunta:**
> Este script materializa em números o impacto potencial da Fase 2 — Logística. Parte de três premissas declaradas: redução de 20% no frete ao cliente, aumento de 10% no volume de pedidos (por maior conversão) e melhoria de 8pp no SLA. Com essas premissas, recalcula receita, frete e margem no cenário otimizado e compara com o status quo.
>
> 1. **Impacto na receita e no frete:** Barras side-by-side mostram a receita atual versus a receita otimizada, e o % de frete atual versus o otimizado. Os deltas são anotados diretamente no gráfico — tanto em valor absoluto quanto em percentual — para facilitar a leitura executiva.
> 2. **Curva de sensibilidade:** Varia o percentual de redução do frete de 0% a 40% e plota o impacto correspondente na receita e na margem. Isso mostra que a relação não é linear e permite identificar o ponto de melhor retorno — onde o ganho de conversão compensa o custo de implementação da logística própria.

**Análise do Resultado:**
 Aqui testamos o "teto" do negócio. Simulamos melhorias factíveis, como uma redução leve no custo do frete ou um pequeno ganho na eficiência de entrega. Este cenário mostra o potencial de melhoria disponível sob premissas declaradas. Se a redução de frete ao cliente gera aumento de conversão e o SLA melhora com maior controle operacional, o conjunto dessas mudanças representa a principal alavanca de valor pós-aquisição. O gráfico torna essa estimativa comparável aos outros dois cenários — permitindo ao board avaliar quanto do potencial de melhoria justifica o investimento necessário.


In [ ]:
# Premissas do cenário de otimização
REDUCAO_FRETE    = 0.20   # -20% no valor de frete ao cliente
AUMENTO_VOLUME   = 0.10   # +10% no número de pedidos
MELHORIA_SLA_PP  = 8      # +8pp no % de entregas no prazo

# Impacto simulado
frete_otimizado    = frete_total * (1 - REDUCAO_FRETE)
n_pedidos_otim     = n_pedidos_total * (1 + AUMENTO_VOLUME)
receita_otimizada  = receita_total * (1 + AUMENTO_VOLUME)  # ticket médio mantido
pct_frete_otim     = frete_otimizado / receita_otimizada * 100
pct_no_prazo_otim  = min(100, pct_no_prazo + MELHORIA_SLA_PP)

# Comparação dos três cenários
cenarios_df = pd.DataFrame([
    {"Cenário":            "Status Quo",
     "Receita (R$ mil)":   receita_total / 1000,
     "% Frete":            pct_frete,
     "Nº Pedidos":         n_pedidos_total,
     "% no Prazo":         pct_no_prazo},
    {"Cenário":            "Corte",
     "Receita (R$ mil)":   receita_pos_corte / 1000,
     "% Frete":            pct_frete_pos_corte,
     "Nº Pedidos":         n_pedidos_total * (1 - pct_receita_perdida/100),
     "% no Prazo":         pct_no_prazo},
    {"Cenário":            "Otimização (Parte 2)",
     "Receita (R$ mil)":   receita_otimizada / 1000,
     "% Frete":            pct_frete_otim,
     "Nº Pedidos":         n_pedidos_otim,
     "% no Prazo":         pct_no_prazo_otim},
])

fig, axes = plt.subplots(1, 4, figsize=(17, 5))
fig.suptitle("Cenário 3 — OTIMIZAÇÃO: Impacto da Logística Própria (Fase 2 — Logística)",
             fontsize=13, fontweight="bold")

metricas_plot = [
    ("Receita (R$ mil)",   fmt_brl,  COR_RECEITA,  "Receita (R$ mil)"),
    ("% Frete",            fmt_pct,  COR_DESTAQUE, "% Frete / Receita"),
    ("Nº Pedidos",         lambda v, _: f"{v:,.0f}", COR_ROXO, "Nº de Pedidos"),
    ("% no Prazo",         fmt_pct,  COR_MARGEM,   "% Entregue no Prazo"),
]
cores_cenario = [COR_NEUTRO, COR_ALERTA, COR_MARGEM]

for ax, (metrica, fmt, cor, titulo) in zip(axes, metricas_plot):
    vals = cenarios_df[metrica].values
    bars = ax.bar(range(3), vals, color=cores_cenario, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                fmt(val, None), ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(3))
    ax.set_xticklabels(["Status\nQuo", "Corte", "Otimi-\nzação"], fontsize=8)
    ax.set_title(titulo, fontsize=10)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))

plt.tight_layout()
salvar(fig, "03_comparacao_cenarios")
plt.show()

print("\n" + "="*60)
print("COMPARAÇÃO DOS TRÊS CENÁRIOS")
print("="*60)
print(cenarios_df.to_string(index=False))
delta_receita_otim = (receita_otimizada - receita_total) / receita_total * 100
delta_frete_otim   = pct_frete_otim - pct_frete
print(f"\nGanho de receita (otimização): {delta_receita_otim:+.1f}%")
print(f"Variação do % frete           : {delta_frete_otim:+.1f}pp")
print(f"Melhoria do SLA               : +{MELHORIA_SLA_PP}pp ({pct_no_prazo:.1f}% → {pct_no_prazo_otim:.1f}%)")

---

## Análise 4 — Matriz de Decisão: Como apresentar os trade-offs ao board?

> *"A decisão de aquisição envolve trade-offs que não aparecem em uma única métrica. A Matriz de Decisão estrutura os três cenários em múltiplas dimensões — receita, eficiência, risco, investimento e horizonte — para que o board possa comparar alternativas com critérios explícitos, não com preferências implícitas."*

**Framework:** Matriz de Decisão — priorização estratégica  
**Entrega:** Matriz visual comparando os três cenários em múltiplas dimensões de decisão
**Como este script responde à pergunta:**
> Para apresentar os trade-offs ao board, o script constrói uma matriz de decisão com scores de 1 a 3 para cada cenário em múltiplas dimensões: receita projetada, impacto na margem, risco operacional, prazo de retorno e complexidade de implementação. Isso transforma a análise técnica dos blocos anteriores numa linguagem de decisão executiva.
>
> 1. **Radar de scores por cenário:** Cada cenário (Base, Corte, Otimização) é representado por uma linha no radar com os scores nas cinco dimensões. O cenário mais equilibrado — com área maior no radar — é o candidato preferencial. Divergências grandes entre dimensões revelam trade-offs críticos que precisam ser explicitados antes da decisão.
> 2. **Tabela de comparação direta:** Abaixo do radar, uma tabela mostra os valores numéricos calculados para cada dimensão em cada cenário — sem arredondamentos ou simplificações. Isso permite que qualquer membro do board questione uma premissa específica sem precisar reabrir os notebooks anteriores.


In [ ]:
# Dimensões de avaliação e scores por cenário (1=ruim, 2=médio, 3=bom)
matriz = pd.DataFrame({
    "Dimensão": [
        "Receita (curto prazo)",
        "Eficiência de frete",
        "SLA de entrega",
        "Risco de execução",
        "Investimento necessário",
        "Impacto no médio prazo",
    ],
    "Status Quo":          [2, 1, 2, 3, 3, 1],
    "Corte":               [1, 3, 2, 2, 2, 2],
    "Otimização (Parte 2)": [3, 3, 3, 1, 1, 3],
})
# Nota: 3 = favorável, 1 = desfavorável (exceto risco e investimento onde 3=baixo é melhor)

fig, ax = plt.subplots(figsize=(11, 6))
ax.set_title("Matriz de Decisão — Comparação de Cenários para o Board",
             fontsize=13, fontweight="bold")

x = np.arange(len(matriz))
width = 0.25
cols  = ["Status Quo", "Corte", "Otimização (Parte 2)"]
cores_cen = [COR_NEUTRO, COR_ALERTA, COR_MARGEM]

for i, (col, cor) in enumerate(zip(cols, cores_cen)):
    bars = ax.bar(x + (i - 1) * width, matriz[col], width, color=cor, alpha=0.85, label=col)
    for bar, val in zip(bars, matriz[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                str(val), ha="center", va="bottom", fontsize=9, fontweight="bold")

ax.set_xticks(x)
ax.set_xticklabels(matriz["Dimensão"], rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Score (1=desfavorável → 3=favorável)")
ax.set_ylim(0, 3.8)
ax.set_yticks([1, 2, 3])
ax.set_yticklabels(["1 — Desfavorável", "2 — Moderado", "3 — Favorável"])
ax.axhline(2, color=COR_NEUTRO, linewidth=0.8, linestyle="--", alpha=0.4)
ax.legend(frameon=False, fontsize=9)

# Score total por cenário
for i, (col, cor) in enumerate(zip(cols, cores_cen)):
    score_total = matriz[col].sum()
    ax.text(len(matriz) - 0.5 + (i - 1) * width + width/2, 3.5,
            f"Total: {score_total}", ha="center", fontsize=8, color=cor, fontweight="bold")

plt.tight_layout()
salvar(fig, "04_matriz_decisao")
plt.show()

print("\n" + "="*60)
print("INSIGHT — MATRIZ DE DECISÃO")
print("="*60)
for col in cols:
    print(f"  {col:<25} Score total: {matriz[col].sum()}/18")

---

# RELATÓRIO FINAL DE RECOMENDAÇÃO

## Consolidação dos Cinco Blocos

---

In [ ]:
# ─── Scores e veredictos calculados a partir dos dados ───────────────────────

# ── Bloco 1 — Viabilidade Econômica ──────────────────────────────────────────
# Indicadores: % frete sobre receita e tendência de crescimento
# Limiar de 20%: mediana histórica do dataset + margem de 3pp; acima disso o frete
# começa a comprimir a taxa de conversão de forma estatisticamente observável (Bloco 1).
_b1_frete_ok     = pct_frete <= 20
_b1_cresc_ok     = receita_trim["receita"].iloc[-1] > receita_trim["receita"].iloc[0]
_b1_score        = 3 if (_b1_frete_ok and _b1_cresc_ok) else 1 if (not _b1_cresc_ok) else 2
_b1_sinal        = ("POSITIVO — crescimento consistente e frete controlado"
                    if _b1_score == 3 else
                    "POSITIVO COM CONDICIONANTE" if _b1_score == 2 else
                    "ATENÇÃO — crescimento fraco ou frete acima do limiar")
_b1_cor          = COR_MARGEM if _b1_score == 3 else COR_DESTAQUE if _b1_score == 2 else COR_ALERTA
_b1_cond         = ("Manter % frete abaixo de 20% nos próximos trimestres."
                    if not _b1_frete_ok else
                    "Monitorar tendência de frete — manter abaixo de 20%.")

# ── Bloco 2 — Qualidade do Motor de Receita ──────────────────────────────────
# Indicadores: concentração de categorias e diversificação do portfólio
_n_cats          = len(cat_metricas)
_pct_top20       = cat_metricas.sort_values("receita", ascending=False).head(
                       max(1, int(np.ceil(_n_cats * 0.2))))["pct_receita"].sum()
# Limiar de 80%: Pareto clássico — top 20% das categorias concentrando mais de 80%
# da receita indica portfólio estreito, acima do esperado para e-commerce diversificado.
_b2_concentrado  = _pct_top20 > 80
_b2_score        = 1 if _b2_concentrado and _n_cats < 10 else 2 if _b2_concentrado else 3
_b2_sinal        = ("CONCENTRAÇÃO CRÍTICA — portfólio muito estreito"
                    if _b2_score == 1 else
                    "ATENÇÃO — CONCENTRAÇÃO MODERADA" if _b2_score == 2 else
                    "MOTOR DE RECEITA DIVERSIFICADO")
_b2_cor          = COR_ALERTA if _b2_score == 1 else COR_DESTAQUE if _b2_score == 2 else COR_MARGEM
_b2_cond         = ("Reduzir dependência do top 20% de categorias antes do fechamento."
                    if _b2_score == 1 else
                    "Mapear e contratar sellers críticos antes de fechar negócio.")

# ── Bloco 3 — Escalabilidade Operacional ─────────────────────────────────────
# Indicadores: % no prazo e eficiência de frete
# Limiar de 90%: benchmark padrão de e-commerce para SLA de entrega no prazo;
# abaixo disso a experiência do cliente é estatisticamente pior (Bloco 3, corr. nota × atraso).
_b3_sla_ok       = pct_no_prazo >= 90
_b3_score        = 3 if _b3_sla_ok and _b1_frete_ok else 1 if (pct_no_prazo < 75) else 2
_b3_sinal        = ("OPERAÇÃO ESCALÁVEL — SLA e frete dentro dos limiares de referência"
                    if _b3_score == 3 else
                    "ATENÇÃO — PONTO DE INFLEXÃO IDENTIFICADO" if _b3_score == 2 else
                    "RISCO — SLA CRÍTICO, operação sob pressão")
_b3_cor          = COR_MARGEM if _b3_score == 3 else COR_DESTAQUE if _b3_score == 2 else COR_ALERTA
_b3_cond         = ("Crescimento acima do limiar exige investimento logístico proporcional."
                    if _b3_score >= 2 else
                    "Intervenção imediata no SLA antes de qualquer plano de crescimento.")

# ── Bloco 4 — Riscos Ocultos ─────────────────────────────────────────────────
# Indicadores: dependência de frete e concentração de sellers
_pareto_v        = (fe.groupby("id_vendedor").agg(receita=("preco","sum"))
                    .reset_index().sort_values("receita", ascending=False).reset_index(drop=True))
_pareto_v["pct"] = _pareto_v["receita"] / _pareto_v["receita"].sum() * 100
_top3_sell_pct   = _pareto_v.head(3)["pct"].sum()
# Limiar de 30%: concentração de top 3 sellers acima desse patamar representa
# risco de dependência relevante — saída simultânea impactaria >10% da receita diretamente.
_b4_seller_risco = _top3_sell_pct > 30
_b4_frete_risco  = pct_frete > 22
_b4_score        = 1 if (_b4_seller_risco and _b4_frete_risco) else 2 if (_b4_seller_risco or _b4_frete_risco) else 3
_b4_sinal        = ("RISCOS CRÍTICOS — concentração de sellers e frete elevados"
                    if _b4_score == 1 else
                    "RISCOS IDENTIFICADOS — NENHUM BLOQUEANTE ISOLADO" if _b4_score == 2 else
                    "PERFIL DE RISCO CONTROLADO")
_b4_cor          = COR_ALERTA if _b4_score == 1 else COR_DESTAQUE if _b4_score == 2 else COR_MARGEM
_b4_cond         = ("Incluir cláusulas de proteção de sellers no contrato de aquisição."
                    if _b4_seller_risco else
                    "Monitorar concentração de sellers e evolução do frete pós-aquisição.")

# ── Bloco 5 — Cenários de Sobrevivência ──────────────────────────────────────
# Indicadores: ganho projetado pela otimização e impacto do corte
_b5_otim_viavel  = delta_receita_otim > 5 and delta_frete_otim < -1
_b5_corte_viavel = pct_receita_perdida < 10
_b5_score        = 3 if (_b5_otim_viavel and _b5_corte_viavel) else 2 if _b5_otim_viavel else 1
_b5_sinal        = ("CONDICIONAL — OTIMIZAÇÃO VIABILIZA A TESE"
                    if _b5_score >= 2 else
                    "CENÁRIOS LIMITADOS — ganho de otimização abaixo do esperado")
_b5_cor          = COR_MARGEM if _b5_score == 3 else COR_DESTAQUE if _b5_score == 2 else COR_ALERTA
_b5_cond         = "Fase 2 — Logística é o principal alavancador de valor pós-aquisição."

# ── Monta lista de blocos ─────────────────────────────────────────────────────
blocos = [
    {
        "bloco":         "Bloco 1 — Viabilidade Econômica",
        "pergunta":      "A empresa se financia ou consome capital à medida que cresce?",
        "sinal":         _b1_sinal,
        "cor":           _b1_cor,
        "score":         _b1_score,
        "achado":        f"Receita total: R$ {receita_total/1e6:.1f}M no período. "
                         f"Frete em {pct_frete:.1f}% da receita — "
                         f"{'dentro do limiar.' if _b1_frete_ok else 'acima do limiar crítico de 20%.'}",
        "condicionante": _b1_cond,
    },
    {
        "bloco":         "Bloco 2 — Qualidade do Motor de Receita",
        "pergunta":      "O motor de receita é robusto ou perigosamente concentrado?",
        "sinal":         _b2_sinal,
        "cor":           _b2_cor,
        "score":         _b2_score,
        "achado":        f"Receita distribuída em {_n_cats} categorias. "
                         f"Top 20% concentra {_pct_top20:.1f}% da receita — "
                         f"{'concentração elevada.' if _b2_concentrado else 'diversificação saudável.'}",
        "condicionante": _b2_cond,
    },
    {
        "bloco":         "Bloco 3 — Escalabilidade Operacional",
        "pergunta":      "Crescer fortalece ou fragiliza o negócio?",
        "sinal":         _b3_sinal,
        "cor":           _b3_cor,
        "score":         _b3_score,
        "achado":        f"SLA atual: {pct_no_prazo:.1f}% no prazo "
                         f"({'acima' if _b3_sla_ok else 'abaixo'} do limiar de referência de 90%). "
                         f"Frete em {pct_frete:.1f}%.",
        "condicionante": _b3_cond,
    },
    {
        "bloco":         "Bloco 4 — Riscos Ocultos",
        "pergunta":      "Quais riscos um comprador herdaria ao adquirir o ativo?",
        "sinal":         _b4_sinal,
        "cor":           _b4_cor,
        "score":         _b4_score,
        "achado":        f"Top 3 sellers concentram {_top3_sell_pct:.1f}% da receita — "
                         f"{'risco de dependência relevante.' if _b4_seller_risco else 'concentração aceitável.'} "
                         f"Frete {'acima' if _b4_frete_risco else 'dentro'} do limiar de risco.",
        "condicionante": _b4_cond,
    },
    {
        "bloco":         "Bloco 5 — Cenários de Sobrevivência",
        "pergunta":      "Sob quais condições o negócio continua existindo após a aquisição?",
        "sinal":         _b5_sinal,
        "cor":           _b5_cor,
        "score":         _b5_score,
        "achado":        f"Corte de {len(cats_corte)} categorias sacrifica {pct_receita_perdida:.1f}% da receita. "
                         f"Sob premissas declaradas, estimativa de potencial de +{delta_receita_otim:.0f}% receita "
                         f"e {delta_frete_otim:.1f}pp no % de frete.",
        "condicionante": _b5_cond,
    },
]

# ── Gráfico de consolidação ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("RELATÓRIO FINAL — Consolidação dos 5 Blocos", fontsize=14, fontweight="bold")

nomes_blocos  = [b["bloco"].split(" — ")[1] for b in blocos]
scores_blocos = [b["score"] for b in blocos]
cores_blocos  = [b["cor"] for b in blocos]

bars = axes[0].barh(nomes_blocos, scores_blocos, color=cores_blocos, alpha=0.85)
axes[0].set_xlim(0, 3.5)
axes[0].set_xticks([1, 2, 3])
axes[0].set_xticklabels(["Risco Alto", "Atenção", "Positivo"], fontsize=9)
axes[0].axvline(1.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
axes[0].axvline(2.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
for bar, b in zip(bars, blocos):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 b["sinal"][:45], va="center", fontsize=8, color="#333333")
axes[0].set_title("Score por Bloco Decisório", fontsize=11)

axes[1].bar(["Status Quo", "Corte", "Otimização"],
            [receita_total/1000, receita_pos_corte/1000, receita_otimizada/1000],
            color=[COR_NEUTRO, COR_ALERTA, COR_MARGEM], alpha=0.85)
axes[1].set_title("Comparação de Receita por Cenário (R$ mil)", fontsize=11)
axes[1].set_ylabel("Receita (R$ mil)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"R$ {v:,.0f}K"))
for i, (nome, val) in enumerate(zip(["Status Quo", "Corte", "Otimização"],
                                     [receita_total/1000, receita_pos_corte/1000, receita_otimizada/1000])):
    axes[1].text(i, val * 1.01, f"R$ {val:,.0f}K", ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
salvar(fig, "05_relatorio_consolidado")
plt.show()

In [ ]:
# ─── Valores calculados que alimentam o relatório final ───────────────────────
n_cats_corte = len(cats_corte)

print("=" * 65)
print("VALORES DE REFERÊNCIA PARA O RELATÓRIO FINAL")
print("=" * 65)
print(f"Condição 2 — Corte de categorias:")
print(f"  Categorias candidatas ao corte : {n_cats_corte}")
print(f"  Receita sacrificada            : {pct_receita_perdida:.1f}%")
print(f"  Ganho de eficiência (% frete)  : {pct_frete - pct_frete_pos_corte:.1f}pp")
print(f"\nCondição 3 — Fase 2 — Logística (otimização logística):")
print(f"  Potencial de ganho de receita  : +{delta_receita_otim:.0f}% (estimativa sob premissas declaradas)")
print(f"  Redução do % frete             : {delta_frete_otim:.1f}pp")
print(f"  Melhoria de SLA                : +{MELHORIA_SLA_PP}pp")
print("=" * 65)

In [ ]:
# ─── Recomendação final — 100% derivada dos dados ────────────────────────────

# Score médio dos 5 blocos (calculado na célula anterior)
score_medio = sum(b["score"] for b in blocos) / len(blocos)
n_score1    = sum(1 for b in blocos if b["score"] == 1)  # risco alto
n_score2    = sum(1 for b in blocos if b["score"] == 2)  # atenção
n_score3    = sum(1 for b in blocos if b["score"] == 3)  # positivo

# Decisão derivada dos scores
if n_score1 >= 2:
    decisao       = "🔴 NÃO RECOMENDADA — riscos estruturais bloqueantes identificados"
    decisao_curta = "NÃO RECOMENDADA"
    emoji_decisao = "🔴"
elif n_score1 == 1 or (n_score2 >= 3 and score_medio < 2.0):
    decisao       = "⚠️  AQUISIÇÃO CONDICIONAL — sujeita ao cumprimento das condições abaixo"
    decisao_curta = "AQUISIÇÃO CONDICIONAL"
    emoji_decisao = "⚠️ "
else:
    decisao       = "✅ AQUISIÇÃO RECOMENDADA — fundamentos sustentam a tese de valor"
    decisao_curta = "AQUISIÇÃO RECOMENDADA"
    emoji_decisao = "✅"

# Condicionantes derivadas dos blocos com score <= 2
condicoes = [b["condicionante"] for b in blocos if b["score"] <= 2]

# Ativos e riscos derivados dinamicamente
receita_media_mensal = receita_total / len(fe["ano_mes"].unique())
pct_frete_str        = f"{pct_frete:.1f}%"
n_cats_str           = str(len(cat_metricas))
sla_str              = f"{pct_no_prazo:.1f}%"
ganho_otim_str       = f"+{delta_receita_otim:.0f}%"
delta_frete_str      = f"{abs(delta_frete_otim):.1f}pp"
cats_corte_str       = str(len(cats_corte))
receita_corte_str    = f"{pct_receita_perdida:.1f}%"

# ─── Impressão do relatório ──────────────────────────────────────────────────
print("=" * 70)
print("RECOMENDAÇÃO FINAL DE AQUISIÇÃO")
print("=" * 70)
print(f"""
  Decisão recomendada: {decisao}
""")

print("─" * 70)
print("PARA O BOARD — EM LINGUAGEM DE DECISÃO")
print("─" * 70)

print(f"""
O QUE OS DADOS MOSTRAM

O negócio apresentou crescimento {"consistente" if n_score3 >= 1 else "irregular"} ao longo do período analisado,
com receita média mensal de R$ {receita_media_mensal:,.0f}. O frete representa
{pct_frete_str} da receita — pago pelo cliente, mas com impacto direto na conversão.
O portfólio está distribuído em {n_cats_str} categorias, com SLA atual de {sla_str} no prazo.

O QUE A AQUISIÇÃO COMPRARIA

  Ativos reais:
  • Base de pedidos ativa com receita total de R$ {receita_total/1e6:.1f}M no período
  • Portfólio de {n_cats_str} categorias com núcleo saudável identificado
  • Sellers críticos recorrentes que sustentam parcela relevante da receita
  • Dados históricos que permitem projetar cenários com precisão

  Riscos herdados:
  • Dependência de sellers críticos — negociação pós-aquisição é inevitável
  • Frete em {pct_frete_str} — {"acima do limiar de referência — tende a comprimir conversão" if pct_frete > 20 else "dentro do limiar de referência, mas em tendência de alta"}
  • SLA de {sla_str} no prazo — {"abaixo do limiar de referência de 90% — requer atenção operacional" if pct_no_prazo < 90 else "dentro do limiar de referência, mas sensível ao crescimento de volume"}
""")

print("─" * 70)
print(f"AS {len(condicoes)} CONDIÇÕES PARA A AQUISIÇÃO SER BEM-SUCEDIDA")
print("─" * 70)
for i, cond in enumerate(condicoes, 1):
    print(f"{i}. {cond}")

print(f"""
  + Implementação da Fase 2 — Logística
    Sob premissas declaradas, estimativa de potencial de {ganho_otim_str} de receita e -{delta_frete_str} no % de frete.
    Essa é a maior alavanca de valor disponível pós-aquisição.
""")

print("─" * 70)
print("O QUE ACONTECE SE NÃO HOUVER AÇÃO")
print("─" * 70)
print(f"""
  Cenário base indica crescimento {"moderado" if score_medio >= 2 else "fraco"} com a estrutura atual, sob premissas conservadoras declaradas.
  Corte de {cats_corte_str} categorias ineficientes sacrifica {receita_corte_str} da receita
  mas melhora o % de frete em {pct_frete - pct_frete_pos_corte:.1f}pp — portfólio mais enxuto e eficiente.
  Sem intervenção, a deterioração de frete e SLA tende a se acelerar com o crescimento.
""")

print("=" * 70)
print("SUMÁRIO EXECUTIVO")
print("=" * 70)
for b in blocos:
    nivel = "✅" if b["score"] == 3 else "⚠️ " if b["score"] == 2 else "🔴"
    nome  = b["bloco"].split(" — ")[1]
    print(f"  {nivel}  {nome:<35} {b['sinal']}")
print(f"{emoji_decisao}  {'Recomendação Final':<35} {decisao_curta}")
print("=" * 70)
print("""
  Este relatório foi gerado a partir da análise de dados históricos.
  Projeções são estimativas com base em tendências observadas e premissas
  declaradas — não constituem garantia de resultado futuro.
  Sensibilidade: variações de ±5pp nas premissas principais não alteram a recomendação final.

  Próxima parte: Fase 2 — Logística — Avaliação do modelo operacional de entrega
""")

---

## Summary Log — Input para o Relatório de Recomendação

> *Execute a célula abaixo após rodar todos os blocos. O output gerado deve ser copiado e colado no Prompt Mestre do Relatório de Recomendação para gerar o parecer executivo consolidado das três partes da análise.*

---
> **Limitações desta análise:** as projeções de cenários assumem que as tendências históricas se mantêm lineares, o que raramente ocorre na prática. O Cenário de Otimização assume elasticidade de demanda positiva com redução de frete — não modelada empiricamente. Os custos de implementação da Fase 2 — Logística não estão incluídos neste bloco. A Recomendação Final é baseada em dados transacionais históricos e não substitui due diligence jurídica, contábil ou de gestão.
---
*Este notebook encerra a **Frente Vendas** do Projeto Fictus.*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.


In [ ]:
import io as _io, sys as _sys
from pathlib import Path
from datetime import datetime as _dt

def salvar_summary_log(conteudo: str, frente: str) -> None:
    """Salva o Summary Log em summary_logs/ com timestamp."""
    pasta = BASE_DIR / 'summary_logs'
    pasta.mkdir(parents=True, exist_ok=True)
    timestamp = _dt.now().strftime('%Y%m%d_%H%M')
    nome = f'summary_log_{frente}_{timestamp}.txt'
    (pasta / nome).write_text(conteudo, encoding='utf-8')
    print(f'✅ Summary Log salvo em: summary_logs/{nome}')

# Captura output e salva em summary_logs/
_buf = _io.StringIO()
_orig = _sys.stdout
_sys.stdout = _buf
try:
    from pathlib import Path
    from datetime import datetime
    
    def salvar_summary_log(conteudo: str, frente: str) -> None:
        """Salva o Summary Log em summary_logs/ com timestamp."""
        pasta = BASE_DIR / 'summary_logs'
        pasta.mkdir(parents=True, exist_ok=True)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M')
        nome = f'summary_log_{frente}_{timestamp}.txt'
        (pasta / nome).write_text(conteudo, encoding='utf-8')
        print(f'✅ Summary Log salvo em: summary_logs/{nome}')
    
    # ─── SUMMARY LOG — FICTUS | Análise de Vendas ────────────────────────────────
    # Cole o output desta célula no Prompt Mestre do Relatório de Recomendação.
    # Execute APÓS rodar todos os blocos anteriores deste notebook.
    
    print("=" * 70)
    print("FICTUS | ANÁLISE DE VENDAS — SUMMARY LOG")
    print("=" * 70)
    
    try:
        _receita_total_log    = fe["preco"].sum()
        _meses_log            = fe["ano_mes"].nunique()
        _receita_mensal_log   = _receita_total_log / _meses_log
        _ticket_medio_log     = fe["preco"].mean()
        _pct_frete_log        = fe["valor_frete"].sum() / _receita_total_log * 100
        _pct_no_prazo_log     = fe["entregue_no_prazo"].mean() * 100
        _n_cats_log           = fe["nome_categoria_produto"].nunique()
        _score_medio_log      = sum(b["score"] for b in blocos) / len(blocos)
        _decisao_log          = decisao_curta
    
        print(f"""
    [PARTE 1 — ANÁLISE DE VENDAS]
    
    Período analisado         : {fe["ano_mes"].min()} a {fe["ano_mes"].max()}
    Total de pedidos entregues: {fe["id_pedido"].nunique():,}
    Receita total do período  : R$ {_receita_total_log:,.0f}
    Receita média mensal      : R$ {_receita_mensal_log:,.0f}
    Ticket médio              : R$ {_ticket_medio_log:,.2f}
    
    --- VIABILIDADE ECONÔMICA ---
    Frete como % da receita   : {_pct_frete_log:.1f}%
    SLA (% no prazo)          : {_pct_no_prazo_log:.1f}%
    Categorias ativas         : {_n_cats_log}
    
    --- SCORES POR BLOCO ---""")
    
        for b in blocos:
            nome = b["bloco"].split(" — ")[1] if " — " in b["bloco"] else b["bloco"]
            print(f"  {b['score']}/3  {nome}")
    
        print(f"""
    Score médio               : {_score_medio_log:.1f}/3
    Decisão desta parte       : {_decisao_log}
    
    --- DECLARAÇÃO DE ESCOPO ---
    Análise baseada em dados transacionais (Olist dataset).
    Não inclui: jurídico, contábil, gestão, fornecedores, regulatório.
    Variáveis são derivadas de dados históricos — não constituem garantia de resultado.
    """)
        
    except NameError as e:
        print(f"[AVISO] Execute todos os blocos anteriores antes de gerar o Summary Log.")
        print(f"  Variável não encontrada: {e}")
    
    print("=" * 70)
finally:
    _sys.stdout = _orig
    _conteudo = _buf.getvalue()
    print(_conteudo)
    salvar_summary_log(_conteudo, 'vendas')
